# Vuln2Action - T5 stage 1 benchmark (1 epoch)

M?c ti?u: ch?y **1 epoch T5-base** tr?n 44.331 train rows ?? ?o th?i gian th?c t?
tr?n Kaggle T4 x2 tr??c khi quy?t ??nh 50 epoch.

Dataset: `minhiw/vuln2action-flow-114560`.
Kh?ng c? exploit source code trong dataset.


In [ ]:
!nvidia-smi
import torch
print("torch:", torch.__version__)
print("cuda devices:", torch.cuda.device_count())


In [ ]:
!rm -rf /kaggle/working/Vuln2Action-Lab
!git clone --branch feature/paper-reproduction https://github.com/Hoang-Minh-Hieu-2806/Vuln2Action-Lab.git /kaggle/working/Vuln2Action-Lab
%cd /kaggle/working/Vuln2Action-Lab
!pip install -q -e ".[train]"


In [ ]:
from pathlib import Path
import glob, os

hits = sorted(glob.glob("/kaggle/input/**/flow_train.jsonl", recursive=True))
print("flow_train hits:", hits)
if not hits:
    raise FileNotFoundError("flow_train.jsonl not found under /kaggle/input")

FLOW_DIR = Path(hits[0]).parent
os.environ["FLOW_DIR"] = str(FLOW_DIR)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("FLOW_DIR:", FLOW_DIR)
for name in ["flow_train.jsonl", "flow_validation.jsonl", "flow_stats.json"]:
    p = FLOW_DIR / name
    print(name, p.exists(), f"{p.stat().st_size / 1024 / 1024:.1f} MiB" if p.exists() else "")


In [ ]:
!python -m torch.distributed.run --nproc_per_node=2 scripts/train_t5.py   --dataset "$FLOW_DIR"   --output-dir /kaggle/working/t5_stage1_benchmark   --model-name t5-base   --target-field target_techniques   --max-input-length 256   --max-target-length 256   --per-device-batch-size 8   --gradient-accumulation-steps 4   --num-train-epochs 1   --save-steps 500   --eval-steps 500   --logging-steps 50   --fp16   --gradient-checkpointing   --auto-resume


In [ ]:
import json
from pathlib import Path
state_path = Path("/kaggle/working/t5_stage1_benchmark/checkpoint-693/trainer_state.json")
if not state_path.exists():
    candidates = sorted(Path("/kaggle/working/t5_stage1_benchmark").glob("checkpoint-*/trainer_state.json"))
    state_path = candidates[-1] if candidates else None
print("state:", state_path)
if state_path:
    state = json.loads(state_path.read_text())
    print("global_step:", state.get("global_step"))
    print("epoch:", state.get("epoch"))
    print("best_metric:", state.get("best_metric"))
    print("last logs:", state.get("log_history", [])[-5:])


## Sau benchmark

Ghi l?i:

- th?i gian 1 epoch
- eval loss cu?i epoch
- GPU memory
- t?c ?? step/gi?y

N?u 1 epoch ch?p nh?n ???c, ch?y ti?p 49 epoch v?i `--auto-resume` v?
`--num-train-epochs 50`. N?u qu? ch?m, gi?m epoch v? ghi r? deviation.
